# 09. 테스트와 디버깅 예제

## Goal

- 정상·오류·경계값 테스트를 설계합니다.
- 실패 메시지로 원인을 좁힙니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

표준 라이브러리 `unittest`로 실행합니다. pytest의 arrange-act-assert 구조에도 같은 원칙을 적용할 수 있습니다.


## Steps

### 테스트 가능한 입력 검증 함수

함수 계약과 테스트 사례를 같은 Notebook에서 비교합니다.


In [1]:
import unittest
from io import StringIO


def parse_port(value: str) -> int:
    if not isinstance(value, str) or not value.isascii() or not value.isdigit():
        raise ValueError("포트는 ASCII 숫자 문자열이어야 합니다")
    port = int(value)
    if not 1 <= port <= 65535:
        raise ValueError("포트 범위 오류")
    return port


class ParsePortTests(unittest.TestCase):
    def test_normal(self):
        self.assertEqual(parse_port("443"), 443)

    def test_boundaries(self):
        self.assertEqual(parse_port("1"), 1)
        self.assertEqual(parse_port("65535"), 65535)

    def test_invalid_values(self):
        for value in ("", "0", "65536", "４４３"):
            with self.subTest(value=value), self.assertRaises(ValueError):
                parse_port(value)


stream = StringIO()
suite = unittest.defaultTestLoader.loadTestsFromTestCase(ParsePortTests)
test_result = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)
print({"tests": test_result.testsRun, "successful": test_result.wasSuccessful()})


{'tests': 3, 'successful': True}


## Checks

테스트 수와 성공 여부를 확인합니다.


In [2]:
assert test_result.testsRun == 3
assert test_result.wasSuccessful()
assert not test_result.failures and not test_result.errors
print("테스트 설계 검사 통과")


테스트 설계 검사 통과


## Next Steps

터미널에서는 실패 테스트만 선택 실행하고 로그·traceback의 최초 원인을 추적합니다.
